In [2]:
import pandas as pd
import numpy as np

import torch
from botorch.models import SingleTaskGP
from botorch.acquisition import qUpperConfidenceBound
from botorch.optim import optimize_acqf_discrete_local_search

from sklearn.preprocessing import MinMaxScaler
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib as mpl
mpl.rcParams['figure.dpi'] = 300

In [3]:
def data_convert(df:pd.DataFrame):
    target_cols =['Nluc template', 'Mg(OAc)2', 'NTPs', 'cap2','HEPES-KOH','pyrophosphatase', 'capped mRNA µg/µL']
    df=df[target_cols]
    col_convert = {'Nluc template':'x0','Mg(OAc)2':'x1','NTPs':'x2','cap2':'x3','HEPES-KOH':'x4','pyrophosphatase':'x5','capped mRNA µg/µL':'y'}
    df = df.rename(columns=col_convert)
    return df

def run_bayes(data:pd.DataFrame,steps:list,maxs:list,mins:list,beta:float=2.0):
    # 実験における変数の数を定義
    num_x = len(data.columns)-1
    # 各変数の刻み幅(step),最小値(min),最大値(max),標準化後の刻み幅(norm_step),標準化後の定義域(norm_range)を定義
    norm_x_steps = []
    norm_x_ranges = []
    for i in range(num_x):
        tmp_step = steps[i]/(maxs[i]-mins[i])
        tmp_range = torch.Tensor(np.arange(0,1+tmp_step,tmp_step))
        norm_x_steps.append(tmp_step)
        norm_x_ranges.append(tmp_range)

    mmscaler = MinMaxScaler()
    norm_x = mmscaler.fit_transform(data.iloc[:,:-1])
    norm_x_df = pd.DataFrame(norm_x)
    y = data.iloc[:,-1].values
    norm_x_df.insert(len(data.columns)-1,'y',-y)
    norm_x_df.columns = data.columns
    norm_x_df
    #norm_x_df.to_csv('./data/round1_normed.csv')
    #norm_data = pd.read_csv("./data/round1_normed.csv", encoding="shift_jis",index_col=0)
    num_x = norm_x_df.shape[1]-1
   
    # 標準化済みのcsvデータ（既知の実験データ）を読み込む
    #norm_data = pd.read_csv("./data/round1_normed.csv", encoding="shift_jis",index_col=0)
    norm_data_x = norm_x_df.iloc[:-2,0:num_x]        #全行のxを読み込む
    norm_data_y = norm_x_df.iloc[:-2,num_x:num_x+1]  #全行のyを読み込む

    X = torch.Tensor(norm_data_x.values)
    Y = -torch.Tensor(norm_data_y.values)

    gp = SingleTaskGP(X, Y)
    qUCB = qUpperConfidenceBound(gp, beta=beta)
    candidate, acq_value = optimize_acqf_discrete_local_search(
        qUCB, discrete_choices=norm_x_ranges,q=20, num_restarts=10, raw_samples=len(data)-2,
    )
    inv_cand = mmscaler.inverse_transform(candidate)
    inv_cand = np.round(inv_cand,3)
    
    return inv_cand

In [4]:
raw_data = pd.read_csv('../data/new_conditions/round_init_exp2.csv',index_col=0).T
raw_data = raw_data[raw_data['Capping efficiency']!=-1]
raw_data  = raw_data.apply(pd.to_numeric, errors='coerce') # convert dtype from object to float
data = data_convert(raw_data)

In [5]:
data

,x0,x1,x2,x3,x4,x5,y
tube1,67.0,56.0,15.0,0.9,27.0,5.0,5.885398
tube3,83.0,56.0,12.0,0.7,55.0,3.0,6.296030
tube5,25.0,98.0,13.0,0.3,60.0,8.0,1.354465
tube6,100.0,20.0,11.0,1.1,44.0,3.0,3.869357
tube9,66.0,25.0,9.0,1.3,52.0,10.0,5.042665
tube10,38.0,120.0,18.0,0.5,30.0,4.0,0.466921


In [6]:
#mins = data.describe().loc['min'].values
#maxs = data.describe().loc['max'].values
#['Nluc template', 'Mg(OAc)2', 'NTPs', 'cap2', 'HEPES-KOH', 'pyrophosphatase', 'capped mRNA µg/µL']

maxs = [100,120,25,1.5,80,10,0]
mins = [10,8,5,0.1,10,1,0]
#maxs = [90,100,6,3,2]
data.loc['min']=mins
data.loc['max']=maxs
# ['Nluc template', 'Mg(OAc)2', 'NTPs', 'cap2', 'HEPES-KOH', 'pyrophosphatase', 'capped mRNA µg/µL']
steps = [0.5,0.5,0.5,0.1,0.5,0.5,0.01] # more detailed step
#steps = [5.0,1.0,1.0,0.5,0.01]

cands = run_bayes(data,steps,maxs,mins)
cands

array([[77.5, 45.5, 12.5,  0.9, 44. ,  5. ],
       [69. , 34.5, 10.5,  1.2, 44. ,  7. ],
       [83.5, 52.5, 11.5,  0.8, 50.5,  2. ],
       [80.5, 41. , 15. ,  0.9, 21.5,  4.5],
       [95. , 44.5, 13. ,  0.7, 57.5,  6. ],
       [61.5, 29.5, 15. ,  0.9, 56. ,  4.5],
       [70.5, 46. ,  6.5,  0.7, 40. ,  4.5],
       [84. , 68. , 12.5,  1.1, 50. ,  4. ],
       [54.5, 58. , 16.5,  1.5, 45. ,  5.5],
       [84. , 21.5, 13.5,  0.4, 43.5,  4.5],
       [55.5, 62. , 12.5,  1.1, 26.5,  8. ],
       [84. , 39.5,  7.5,  1.2, 30.5,  5.5],
       [86.5, 65. , 19.5,  0.7, 22.5,  7.5],
       [72.5, 36.5, 17. ,  1.3, 37. ,  2. ],
       [78. , 24. ,  8.5,  0.6, 50.5,  8.5],
       [97.5, 15.5, 14. ,  1.1, 57. ,  5.5],
       [91.5, 45.5, 17. ,  1.2, 29.5,  6.5],
       [87.5, 19. ,  7. ,  1.1, 58. ,  7.5],
       [73.5, 43.5, 14. ,  0.5, 34.5,  5.5],
       [57.5, 32. , 14. ,  1.2, 44. ,  4.5]])

In [7]:
index = [f'cand{i}' for i in range(1,len(cands)+1)]
cand_df = pd.DataFrame(cands,index=index,columns=data.columns[:-1])
col_convert = {'Nluc template':'x0','Mg(OAc)2':'x1','NTPs':'x2','cap2':'x3','HEPES-KOH':'x4','pyrophosphatase':'x5','capped mRNA µg/µL':'y'}
col_conv_rev = {v:k for k,v in col_convert.items()}
cand_df=cand_df.rename(columns=col_conv_rev)
cand_df.to_csv('../outputs/round1_cand_qUCB_beta2_valid_samples.csv')

In [8]:
cand_df

,Nluc template,Mg(OAc)2,NTPs,cap2,HEPES-KOH,pyrophosphatase
cand1,77.5,45.5,12.5,0.9,44.0,5.0
cand2,69.0,34.5,10.5,1.2,44.0,7.0
cand3,83.5,52.5,11.5,0.8,50.5,2.0
cand4,80.5,41.0,15.0,0.9,21.5,4.5
cand5,95.0,44.5,13.0,0.7,57.5,6.0
cand6,61.5,29.5,15.0,0.9,56.0,4.5
cand7,70.5,46.0,6.5,0.7,40.0,4.5
cand8,84.0,68.0,12.5,1.1,50.0,4.0
cand9,54.5,58.0,16.5,1.5,45.0,5.5
cand10,84.0,21.5,13.5,0.4,43.5,4.5
